In [1]:
# # Import packages
import cv2
import numpy as np
import math

In [3]:
# image version - lines to edge of image

# Load image, grayscale, Otsu's threshold
image = cv2.imread('worm_draw_dots.png')
imageS = cv2.resize(image, (340, 560)) 
gray = cv2.cvtColor(imageS, cv2.COLOR_BGR2GRAY)
thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1] # cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

# Morph open using elliptical shaped kernel
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=3)

# Find circles 
cnts = cv2.findContours(opening, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cnts = cnts[0] if len(cnts) == 2 else cnts[1]
circles = []
center = []
for c in cnts:
    area = cv2.contourArea(c)
    if area > 100 and area < 300:
        ((x, y), r) = cv2.minEnclosingCircle(c)
        # Get center coordinates and radii of all circles (converted to int)
        circles.append([cv2.minEnclosingCircle(c)])
        circle = cv2.minEnclosingCircle(c) 
        center.append([np.int32(np.round(circle[0]))])
        cv2.circle(imageS, (int(x), int(y)), int(r), (36, 255, 12), 2)

# draw lines between circles

# Find horizontal lines within tolerance; calculate mean
centers = np.array(center)
n = len(centers)
tol = 30
x_match = np.array([np.abs(centers[:,0,1] - cent[0,1]) < tol for cent in centers])
lines_global = []
lines_start = []
lines_end = []
for i in np.arange(n):
    lines_local = []
    lines_local.append(i)
    for j in np.arange(i+1, n):
        if (x_match[i, j]):
            lines_local.append(j)
    if (len(lines_local) > 1):
        lines_global.append(np.int32(np.round(np.mean(centers[lines_local,0,1]))))

        # find max and min value
        low = centers[lines_local,0,0].argmin()
        high = centers[lines_local,0,0].argmax()
        lines_start.append(centers[lines_local[low],0])
        lines_end.append(centers[lines_local[high],0])
        for j in lines_local:
            for k in lines_local:
                x_match[j, k] = False
                x_match[k, j] = False

# Draw horizontal lines
i = 0
for line in lines_global:
    cv2.line(imageS, (lines_start[i][0], line), (lines_end[i][0], line), (255, 0, 0), 2)
    i = i + 1

# Find vertical lines within tolerance; calculate mean
x_match = np.array([np.abs(centers[:, 0,0] - cent[0,0]) < tol for cent in centers])
lines_global = []
for i in np.arange(n):
    lines_local = []
    lines_local.append(i)
    for j in np.arange(i+1, n):
        if (x_match[i, j]):
            lines_local.append(j)
    if (len(lines_local) > 1):
        lines_global.append(np.int32(np.round(np.mean(centers[lines_local, 0,0]))))
        for j in lines_local:
            for k in lines_local:
                x_match[j, k] = False
                x_match[k, j] = False

# Draw vertical lines
for line in lines_global:
    cv2.line(imageS, (line, 0), (line, imageS.shape[0]), (255, 0, 0), 2)                

cv2.imshow('thresh', thresh)
cv2.waitKey()
cv2.imshow('opening', opening)
cv2.waitKey()
cv2.imshow('image', imageS)
cv2.waitKey()
cv2.destroyAllWindows()

In [2]:
# image version - squares between circles and spine

# Load image, grayscale, Otsu's threshold
image = cv2.imread('worm_draw_dots.png')
imageS = cv2.resize(image, (340, 560)) 
gray = cv2.cvtColor(imageS, cv2.COLOR_BGR2GRAY)
thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1] # inverse converts background to black and lines/dots to white

# Morph open using elliptical shaped kernel
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=3)

# Find circles 
cnts = cv2.findContours(opening, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cnts = cnts[0] if len(cnts) == 2 else cnts[1]
circles = []
center = []
for c in cnts:
    area = cv2.contourArea(c)
    if area > 100 and area < 300:
        print(area)
        ((x, y), r) = cv2.minEnclosingCircle(c)
        # Get center coordinates and radii of all circles (converted to int)
        circles.append([cv2.minEnclosingCircle(c)])
        circle = cv2.minEnclosingCircle(c) 
        center.append([np.int32(np.round(circle[0]))])
        cv2.circle(imageS, (int(x), int(y)), int(r), (36, 255, 12), 2)

# draw squares between circles

# Find nearest neighbours
centers = np.array(center)
n = len(centers)
tol = 30 # may have to increase tolerance when bending?

x_match = np.array([np.abs(centers[:,0,0] - cent[0,0]) < tol for cent in centers]) # vertical lines
y_match = np.array([np.abs(centers[:,0,1] - cent[0,1]) < tol for cent in centers]) # horizontal lines

# finding the shortest vertical lines so you don't get the ones that just go straight across  
track_dist = []
track_i = []
track_j = []

for i in np.arange(n):
    for j in np.arange(i+1,n):
        if (x_match[i, j]):
            track_dist.append(abs(centers[i,0,1] - centers[j,0,1]))
            track_i.append(i)
            track_j.append(j)

sort_dist = sorted(zip(track_dist,track_i,track_j))

# keep shortest 4 distances (4 will change for actual actuator)
# sort_dist = sort_dist[:-2]
for m in np.arange(10, len(sort_dist)):  
    x_match[sort_dist[m][1], sort_dist[m][2]] = False

# draw lines for squares on image
# use centre of horizontal lines for the spine
spine_coords = []
for i in np.arange(n): # i is centers[i]

    for j in np.arange(i+1, n):
        if (y_match[i, j]):
            start_hy = centers[i,0,1]
            end_hy = centers[j,0,1]
            start_hx = centers[i,0,0]
            end_hx = centers[j,0,0]
            cv2.line(imageS, (start_hx, start_hy), (end_hx, end_hy), (255, 0, 0), 2)

            # use centre of horizontal lines for the spine
            spine_coords.append([math.floor((start_hx + end_hx)/2), math.floor((start_hy + end_hy)/2)])
 
        if (x_match[i, j]):
            start_vx = centers[i,0,0]
            end_vx = centers[j,0,0]
            start_vy = centers[i,0,1]
            end_vy = centers[j,0,1]
            cv2.line(imageS, (start_vx, start_vy), (end_vx, end_vy), (255, 0, 0), 2)

    # draw spine
    for s in range(len(spine_coords)-1):
        cv2.line(imageS, (spine_coords[s][0],spine_coords[s][1]), (spine_coords[s+1][0],spine_coords[s+1][1]), (0, 0, 255), 2)
 
    for s in range(len(spine_coords)-2):
        p1x = spine_coords[s][0]
        p1y = spine_coords[s][1]

        p2x = spine_coords[s+1][0]
        p2y = spine_coords[s+1][1]

        p3x = spine_coords[s+2][0]
        p3y = spine_coords[s+2][1]

        opp1 = abs(p1y - p2y)
        opp2 = abs(p3y - p2y)

        adj1 = abs(p1x - p2x)
        adj2 = abs(p3x - p2x)

        if (adj1 != 0):
            a = math.tan(opp1/adj1)
        else:
            a = 0

        if (adj2 != 0):
            b = math.tan(opp2/adj2)
        else:
            b = 0

        if (p3y > p2y) and (p1y > p2y):
            angle = (math.pi - a) + b
        elif (p3y < p2y) and (p1y < p2y):
            angle = (math.pi - b) + a
        else:
            angle = math.pi - a - b
            angle = angle * 180/math.pi

        print("angle: ", angle)

cv2.imshow('thresh', thresh)
cv2.waitKey()
cv2.imshow('opening', opening)
cv2.waitKey()
cv2.imshow('image', imageS)
cv2.waitKey()
cv2.destroyAllWindows()

159.0
159.0
156.5
157.5
154.5
156.0
153.5
154.5
148.0
153.5
154.0
156.5
angle:  187.5291981927915
angle:  187.5291981927915
angle:  187.5291981927915
angle:  13156.675351953845
angle:  187.5291981927915
angle:  13156.675351953845
angle:  187.5291981927915
angle:  13156.675351953845
angle:  13126.029879233884
angle:  187.5291981927915
angle:  13156.675351953845
angle:  13126.029879233884
angle:  187.5291981927915
angle:  13156.675351953845
angle:  13126.029879233884
angle:  160.68014831982973
angle:  187.5291981927915
angle:  13156.675351953845
angle:  13126.029879233884
angle:  160.68014831982973


In [4]:
# find dots and draw squares between dots

# sets the video source to the default webcam, which OpenCV can easily capture.
video_capture = cv2.VideoCapture(0)



while True:
    # Capture frame-by-frame - read() function reads one frame from the video source, which in this example is the webcam. 
    # This returns: (1) actual video frame read (one frame on each loop) and (2) a return code (return code tells us if we 
    # have run out of frames, which will happen if we are reading from a file. This doesn’t matter when reading from the 
    # webcam, since we can record forever, so we will ignore it.
    ret, frame = video_capture.read()

    # convert to greyscale. Alternative methods could include visual processing using the HSI scale (talk about how a mask could be 
    # used for specific colours and how this would be especially advantageous for tracking of multiple segments)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Apply otsu's threshold (define?) and invert binary scale so that anything above the threshold (edges and the dots) are white and 
    # the rest of the image is black. This makes visual processing easier as there are now only 2 levels rather than the 256 of 
    # greyscale. 
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1] 
    
    # Morph open using elliptical shaped kernel
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=3)
    
    # Find circles 
    # findContours() detects changes in image colour and marks these as contours
    cnts = cv2.findContours(opening, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = cnts[0] if len(cnts) == 2 else cnts[1]
    center = []
    for c in cnts:
        area = cv2.contourArea(c)

        # the area of the contour must be within the range expected for the dot drawn on the actuator.
        # If it is, a circle is drawn around the contour.
        if area > 50 and area < 1000:
            ((x, y), r) = cv2.minEnclosingCircle(c)
            circle = cv2.minEnclosingCircle(c) 
            center.append([np.int32(np.round(circle[0]))])
            cv2.circle(frame, (int(x), int(y)), int(r), (36, 255, 12), 2)

    # draw squares between circles to create a skeletal structure of the worm

    centers = np.array(center) # convert to numpy
    n = len(centers) # calculate number of circles 
    tol = 30 # may have to increase tolerance when bending?

    # compare coordinates of dots, if an x or y coordinate is similar mark it as True
    x_match = np.array([np.abs(centers[:,0,0] - cent[0,0]) < tol for cent in centers]) # vertical lines
    y_match = np.array([np.abs(centers[:,0,1] - cent[0,1]) < tol for cent in centers]) # horizontal lines

    # finding the shortest vertical lines so you don't get the ones that just go straight across  
    # depending on orientation of actual worm video you may have to switch this to horizontal
    track_dist = []
    track_i = []
    track_j = []

    # create an ordered list of all the vertical lines in current frame
    for i in np.arange(n):
        for j in np.arange(i+1,n):
            if (x_match[i, j]):
                track_dist.append(abs(centers[i,0,1] - centers[j,0,1]))
                track_i.append(i)
                track_j.append(j)
    sort_dist = sorted(zip(track_dist,track_i,track_j))

    # keep shortest n distances (4 for image, will change for actual actuator)
    for m in np.arange(4, len(sort_dist)):  
        x_match[sort_dist[m][1], sort_dist[m][2]] = False

    # draw lines for squares on image
    spine_coords = []
    for i in np.arange(n): # i is centers[i]
        for j in np.arange(i+1, n):
            # horizontal lines
            if (y_match[i, j]):
                start_hy = centers[i,0,1]
                end_hy = centers[j,0,1]
                start_hx = centers[i,0,0]
                end_hx = centers[j,0,0]
                cv2.line(frame, (start_hx, start_hy), (end_hx, end_hy), (255, 0, 0), 2)

                # use centre of horizontal lines for the spine
                spine_coords.append([math.floor((start_hx + end_hx)/2), math.floor((start_hy + end_hy)/2)])

            # vertical lines        
            if (x_match[i, j]):
                start_vx = centers[i,0,0]
                end_vx = centers[j,0,0]
                start_vy = centers[i,0,1]
                end_vy = centers[j,0,1]
                cv2.line(frame, (start_vx, start_vy), (end_vx, end_vy), (255, 0, 0), 2)

    # draw spine
    for s in range(len(spine_coords)-1):
        cv2.line(frame, (spine_coords[s][0],spine_coords[s][1]), (spine_coords[s+1][0],spine_coords[s+1][1]), (0, 0, 255), 2)

    # display video with respective lines and circles drawn on actuator.
    cv2.imshow('image', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# When everything is done, release the capture
video_capture.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

In [ ]:
# plot line through centre of all squares

In [ ]:
# import cv2


# def cam_test(port: int = 0) -> None:
#     cap = cv2.VideoCapture(port)
#     if not cap.isOpened():  # Check if the web cam is opened correctly
#         print("failed to open cam")
#     else:
#         print('cam opened on port {}'.format(port))

#         for i in range(10 ** 10):
#             success, cv_frame = cap.read()
#             if not success:
#                 print('failed to capture frame on iter {}'.format(i))
#                 break
#             cv2.imshow('Input', cv_frame)
#             k = cv2.waitKey(1)
#             if k == ord('q'):
#                 break

#         cap.release()
#         cv2.destroyAllWindows()
#     return


# if __name__ == '__main__':
#     cam_test()

cam opened on port 0
failed to capture frame on iter 0
